# 8st week's homework - by [Aleksei Novikov](https://www.linkedin.com/in/devnovikov/)

## Preparation

In [25]:
import torch
from PIL import Image
import numpy as np
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms

import os
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.optim as optim

from torchinfo import summary

In [38]:
import numpy as np
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available():
  torch.mps.manual_seed(SEED)

torch.use_deterministic_algorithms(True, warn_only=True)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [39]:
device = torch.device(
  "cuda" if torch.cuda.is_available()
  else "mps" if torch.backends.mps.is_available()
  else "cpu"
)

In [40]:
class HairDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = sorted(os.listdir(data_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        for label_name in self.classes:
            if label_name.startswith('.'):
                continue
            label_dir = os.path.join(data_dir, label_name)
            if not os.path.isdir(label_dir):
                continue
            for img_name in os.listdir(label_dir):
                if img_name.startswith('.'):
                    continue
                self.image_paths.append(os.path.join(label_dir, img_name))
                self.labels.append(self.class_to_idx[label_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [41]:
class HairClassifier(nn.Module):
    def __init__(self):
        super(HairClassifier, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()

        self.flatten = nn.Flatten()
        self.flatten_size = 32 * 99 * 99

        self.fc1 = nn.Linear(self.flatten_size, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = HairClassifier().to(device)

## Homework

## Question 1: Loss Function

**Answer: `nn.BCEWithLogitsLoss()`**


## Question 2: Total Number of Parameters

**Conv2d(3, 32, kernel_size=3):**
- Weights: 3 × 32 × 3 × 3 = 864
- Bias: 32
- Total: **896**

**Linear(313632, 64):**
- Weights: 313,632 × 64 = 20,072,448
- Bias: 64
- Total: **20,072,512**

**Linear(64, 1):**
- Weights: 64 × 1 = 64
- Bias: 1
- Total: **65**

**Grand Total: 896 + 20,072,512 + 65 = 20,073,473**

**Answer: 20,073,473**

In [42]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

summary(model, input_size=(1, 3, 200, 200))

Total parameters: 20073473


Layer (type:depth-idx)                   Output Shape              Param #
HairClassifier                           [1, 1]                    --
├─Conv2d: 1-1                            [1, 32, 198, 198]         896
├─ReLU: 1-2                              [1, 32, 198, 198]         --
├─MaxPool2d: 1-3                         [1, 32, 99, 99]           --
├─Flatten: 1-4                           [1, 313632]               --
├─Linear: 1-5                            [1, 64]                   20,072,512
├─ReLU: 1-6                              [1, 64]                   --
├─Linear: 1-7                            [1, 1]                    65
Total params: 20,073,473
Trainable params: 20,073,473
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 55.20
Input size (MB): 0.48
Forward/backward pass size (MB): 10.04
Params size (MB): 80.29
Estimated Total Size (MB): 90.81

In [43]:
train_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )  # ImageNet normalization
])

test_transforms = train_transforms

In [44]:
data_dir = './data'

train_dataset = HairDataset(
    data_dir=f'{data_dir}/train',
    transform=train_transforms
)

validation_dataset = HairDataset(
    data_dir=f'{data_dir}/test',
    transform=test_transforms
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(validation_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=20, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(validation_loader)}")

Training samples: 801
Validation samples: 201
Training batches: 41
Validation batches: 11


In [45]:
model = model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

criterion = nn.BCEWithLogitsLoss()


In [46]:
# Training loop for 10 epochs (without augmentation)
num_epochs = 10
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1)  # Shape: (batch_size, 1)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        # Apply sigmoid and threshold for predictions
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)
    
    # Validation phase
    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
    
    val_epoch_loss = val_running_loss / len(validation_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

Epoch 1/10, Loss: 0.6325, Acc: 0.6454, Val Loss: 0.5960, Val Acc: 0.6517
Epoch 2/10, Loss: 0.5531, Acc: 0.6816, Val Loss: 0.6535, Val Acc: 0.6318
Epoch 3/10, Loss: 0.5172, Acc: 0.7241, Val Loss: 0.6622, Val Acc: 0.6368
Epoch 4/10, Loss: 0.4791, Acc: 0.7341, Val Loss: 0.6391, Val Acc: 0.6567
Epoch 5/10, Loss: 0.4495, Acc: 0.7541, Val Loss: 0.6494, Val Acc: 0.7065
Epoch 6/10, Loss: 0.3940, Acc: 0.8127, Val Loss: 0.6532, Val Acc: 0.6667
Epoch 7/10, Loss: 0.3211, Acc: 0.8514, Val Loss: 0.8741, Val Acc: 0.6269
Epoch 8/10, Loss: 0.3697, Acc: 0.8315, Val Loss: 0.6747, Val Acc: 0.6965
Epoch 9/10, Loss: 0.2224, Acc: 0.9051, Val Loss: 0.7246, Val Acc: 0.6915
Epoch 10/10, Loss: 0.2001, Acc: 0.9263, Val Loss: 0.8026, Val Acc: 0.6517


## Question 3: Median of Training Accuracy

What is the median of training accuracy for all the epochs?

Answer: 0.84

In [47]:
train_accuracies = history['acc']
median_acc = np.median(train_accuracies)

print(f"Training accuracies each epoch:")
for i, acc in enumerate(train_accuracies, 1):
    print(f"  Epoch {i}: {acc:.4f}")

print(f"Sorted vlaues: {sorted(train_accuracies)}")
print(f"Median: {median_acc:.4f}")

options = [0.05, 0.12, 0.40, 0.84]
closest = min(options, key=lambda x: abs(x - median_acc))
print(f"Closest Answer: {closest}")

Training accuracies each epoch:
  Epoch 1: 0.6454
  Epoch 2: 0.6816
  Epoch 3: 0.7241
  Epoch 4: 0.7341
  Epoch 5: 0.7541
  Epoch 6: 0.8127
  Epoch 7: 0.8514
  Epoch 8: 0.8315
  Epoch 9: 0.9051
  Epoch 10: 0.9263
Sorted vlaues: [0.6454431960049938, 0.6816479400749064, 0.7240948813982522, 0.7340823970037453, 0.7540574282147315, 0.8127340823970037, 0.8314606741573034, 0.8514357053682896, 0.9051186017478152, 0.9263420724094882]
Median: 0.7834
Closest Answer: 0.84


## Question 4: Standard Deviation of Training Loss

What is the standard deviation of training loss for all the epochs?

Answer: 0.171

In [50]:
std_train_loss = np.std(history['loss'])
print(f"Training losses: {[f'{loss:.4f}' for loss in history['loss']]}")
print(f"Standard deviation of training loss: {std_train_loss:.4f}")

options = [0.007, 0.078, 0.171, 1.710]
closest = min(options, key=lambda x: abs(x - std_train_loss))

print(f"Closest Answer: {closest:.3f}")

Training losses: ['0.6325', '0.5531', '0.5172', '0.4791', '0.4495', '0.3940', '0.3211', '0.3697', '0.2224', '0.2001']
Standard deviation of training loss: 0.1330
Closest Answer: 0.171


## Data Augmentation

Data augmentations and train for 10 more epochs without recreating model.

In [51]:
train_transforms_aug = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.RandomRotation(50),
    transforms.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset_aug = HairDataset(
    data_dir=f'{data_dir}/train',
    transform=train_transforms_aug
)

train_loader = DataLoader(train_dataset_aug, batch_size=20, shuffle=True)

print(f"Training dataset with augmentation: {len(train_dataset_aug)} samples")

Training dataset with augmentation: 801 samples


In [52]:
num_epochs = 10
history_aug = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(train_dataset_aug)
    epoch_acc = correct_train / total_train
    history_aug['loss'].append(epoch_loss)
    history_aug['acc'].append(epoch_acc)
    
    # Validation phase
    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
    
    val_epoch_loss = val_running_loss / len(validation_dataset)
    val_epoch_acc = correct_val / total_val
    history_aug['val_loss'].append(val_epoch_loss)
    history_aug['val_acc'].append(val_epoch_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

Epoch 1/10, Loss: 0.6186, Acc: 0.6604, Val Loss: 0.6180, Val Acc: 0.6915
Epoch 2/10, Loss: 0.6045, Acc: 0.6529, Val Loss: 0.7792, Val Acc: 0.6269
Epoch 3/10, Loss: 0.5899, Acc: 0.6717, Val Loss: 0.5405, Val Acc: 0.6915
Epoch 4/10, Loss: 0.5911, Acc: 0.6667, Val Loss: 0.5333, Val Acc: 0.7214
Epoch 5/10, Loss: 0.5433, Acc: 0.7079, Val Loss: 0.5996, Val Acc: 0.6965
Epoch 6/10, Loss: 0.5275, Acc: 0.7141, Val Loss: 0.5294, Val Acc: 0.7313
Epoch 7/10, Loss: 0.5536, Acc: 0.7054, Val Loss: 0.5670, Val Acc: 0.6965
Epoch 8/10, Loss: 0.5326, Acc: 0.7291, Val Loss: 0.5942, Val Acc: 0.6915
Epoch 9/10, Loss: 0.5025, Acc: 0.7316, Val Loss: 0.5122, Val Acc: 0.7413
Epoch 10/10, Loss: 0.4921, Acc: 0.7653, Val Loss: 0.5952, Val Acc: 0.6965


## Question 5: Mean of Test Loss (with Augmentation)

What is the mean of test loss for all the epochs for the model trained with augmentations?

Answer: 0.88

In [55]:
mean_val_loss_aug = np.mean(history_aug['val_loss'])
print(f"Test losses: {[f'{loss:.4f}' for loss in history_aug['val_loss']]}")
print(f"Mean test loss (with augmentation): {mean_val_loss_aug:.4f}")

options = [0.008, 0.08, 0.88, 8.88]
closest = min(options, key=lambda x: abs(x - mean_val_loss_aug))
print(f"Closest Answer: {closest}")

Test losses: ['0.6180', '0.7792', '0.5405', '0.5333', '0.5996', '0.5294', '0.5670', '0.5942', '0.5122', '0.5952']
Mean test loss (with augmentation): 0.5869
Closest Answer: 0.88


## Question 6: Average Test Accuracy (Last 5 Epochs)

What's the average of test accuracy for the last 5 epochs (from 6 to 10) for the model trained with augmentations?

In [58]:
last_5_val_acc = history_aug['val_acc'][-5:]  # Last 5 epochs
avg_val_acc_last5 = np.mean(last_5_val_acc)
print(f"Test accuracies (all 10 epochs): {[f'{acc:.4f}' for acc in history_aug['val_acc']]}")
print(f"Test accuracies (last 5 epochs): {[f'{acc:.4f}' for acc in last_5_val_acc]}")
print(f"Average test accuracy (last 5 epochs): {avg_val_acc_last5:.4f}")

options = [0.08, 0.28, 0.68, 0.98]
closest = min(options, key=lambda x: abs(x - avg_val_acc_last5))
print(f"Closest Answer: {closest}")



Test accuracies (all 10 epochs): ['0.6915', '0.6269', '0.6915', '0.7214', '0.6965', '0.7313', '0.6965', '0.6915', '0.7413', '0.6965']

Test accuracies (last 5 epochs): ['0.7313', '0.6965', '0.6915', '0.7413', '0.6965']

Average test accuracy (last 5 epochs): 0.7114
Closest Answer: 0.68


## Summary of Answers

| Question | Answer |
|----------|--------|
| Q1: Loss function | `nn.BCEWithLogitsLoss()` |
| Q2: Total parameters | 20,073,473 |
| Q3: Median training accuracy | ~0.40 |
| Q4: Std training loss | ~0.171 |
| Q5: Mean test loss (augmented) | ~0.88 |
| Q6: Avg test accuracy (last 5) | ~0.68 |